# Notebook 03 — Sparse Feature Analogy

**Purpose:** build toy sparse-feature matrices over Mod30 residue lanes.

Notebook 01: Mod30 residue manifold.  
Notebook 02: one global-style detector vs local tile union.  
Notebook 03: explicit sparse-feature representations.

Core claim:

```text
One detector is sparse and readable but incomplete.
One-hot local tiles are fragmented but complete.
Grouped tiles trade compactness for coverage.
```

This gives a finite arithmetic analogue for SAE-style local tiling and feature dilution.


## 0. Bulletproof setup

Run first.

In [ ]:

from pathlib import Path
import sys

def find_repo_root(start=None, marker="src"):
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    return None

REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "mod30-manifold-tiling"
    SRC_DIR = REPO_ROOT / "src"
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    (SRC_DIR / "__init__.py").write_text("", encoding="utf-8")
    (SRC_DIR / "mod30.py").write_text("""from math import gcd
MOD30 = 30
MOD30_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
def mod_index(n, mod): return n % mod
def mod_mask(n, residues, mod): return mod_index(n, mod) in residues
def generate_coprime_residues(mod): return [r for r in range(1, mod) if gcd(r, mod) == 1]
def mod30_index(n): return mod_index(n, MOD30)
def mod30_mask(n): return mod_mask(n, MOD30_RESIDUES, MOD30)
def mod30_residues(n_max): return [n for n in range(2, n_max) if mod30_mask(n)]
def single_lane_mask(n, lane=1): return mod30_index(n) == lane
""", encoding="utf-8")
    (SRC_DIR / "tiling_metrics.py").write_text("""def lane_coverage(captured_residues, target_residues):
    captured, target = set(captured_residues), set(target_residues)
    hit, missed = captured & target, target - captured
    return {"target_lanes": len(target), "captured_lanes": len(hit), "missed_lanes": len(missed),
            "coverage_fraction": len(hit)/len(target) if target else 0.0,
            "captured_residues": sorted(hit), "missed_residues": sorted(missed)}
def feature_metrics(feature_df, target_residues, residue_col="mod30_residue"):
    rows = []
    feature_cols = [c for c in feature_df.columns if c.startswith("F_")]
    for col in feature_cols:
        captured = sorted(feature_df.loc[feature_df[col] == 1, residue_col].unique())
        cov = lane_coverage(captured, target_residues)
        rows.append({"feature": col, "activation_fraction": float(feature_df[col].mean()),
                     "captured_lanes": cov["captured_lanes"], "coverage_fraction": cov["coverage_fraction"],
                     "captured_residues": cov["captured_residues"], "missed_residues": cov["missed_residues"]})
    return rows
""", encoding="utf-8")
    (SRC_DIR / "plots.py").write_text("""import matplotlib.pyplot as plt
def save_current(path, dpi=180):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    return path
""", encoding="utf-8")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"
for d in [FIGURES_DIR, DATA_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("src exists:", (REPO_ROOT / "src").exists())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.mod30 import MOD30_RESIDUES, mod30_index, mod30_mask
from src.tiling_metrics import lane_coverage, feature_metrics
from src.plots import save_current

print("Persisting Mod30 lanes:", MOD30_RESIDUES)


## 1. Build Mod30 dataset

We use `n = 1..300`, matching Notebooks 01–02.


In [ ]:
n_min = 1
n_max = 300
values = np.arange(n_min, n_max + 1)

df = pd.DataFrame({
    "n": values,
    "mod30_residue": [mod30_index(int(n)) for n in values],
})
df["inside_mod30_gate"] = df["n"].apply(lambda n: mod30_mask(int(n)))

df.head(12)


## 2. Feature family A — single global-style detector

This deliberately uses one residue lane, `residue == 1`, as a narrow detector.

It is interpretable, but it misses seven persisting lanes.


In [ ]:
single_df = df[["n", "mod30_residue", "inside_mod30_gate"]].copy()
single_df["F_single_lane_r1"] = (single_df["mod30_residue"] == 1).astype(int)

single_metrics = pd.DataFrame(feature_metrics(single_df, MOD30_RESIDUES))
single_metrics


## 3. Feature family B — eight one-hot local tile detectors

Each persisting residue lane gets its own sparse detector.


In [ ]:
onehot_df = df[["n", "mod30_residue", "inside_mod30_gate"]].copy()

for r in MOD30_RESIDUES:
    onehot_df[f"F_tile_r{r}"] = (onehot_df["mod30_residue"] == r).astype(int)

onehot_metrics = pd.DataFrame(feature_metrics(onehot_df, MOD30_RESIDUES))
onehot_metrics


## 4. Feature family C — grouped residue detectors

Grouped features trade fragmentation for compactness.

Here we use four grouped detectors, each covering two persisting lanes.


In [ ]:
groups = {
    "F_group_A_r1_7": [1, 7],
    "F_group_B_r11_13": [11, 13],
    "F_group_C_r17_19": [17, 19],
    "F_group_D_r23_29": [23, 29],
}

grouped_df = df[["n", "mod30_residue", "inside_mod30_gate"]].copy()

for feature, residues in groups.items():
    grouped_df[feature] = grouped_df["mod30_residue"].isin(residues).astype(int)

grouped_metrics = pd.DataFrame(feature_metrics(grouped_df, MOD30_RESIDUES))
grouped_metrics


## 5. Representation-level metrics

We summarize each representation as a family.

Definitions:

```text
coverage_fraction = persisting lanes captured / 8
feature_count = number of detectors
mean_feature_activation = mean activation across feature columns
dilution_index = feature_count / captured_lanes
```

`dilution_index` is intentionally simple: how many feature slots are used per covered lane.


In [ ]:
def representation_summary(name, feature_df):
    feature_cols = [c for c in feature_df.columns if c.startswith("F_")]
    captured_residues = set()
    for col in feature_cols:
        captured_residues.update(feature_df.loc[feature_df[col] == 1, "mod30_residue"].unique().tolist())
    cov = lane_coverage(captured_residues, MOD30_RESIDUES)
    mean_feature_activation = float(feature_df[feature_cols].mean().mean()) if feature_cols else 0.0
    active_rows_fraction = float((feature_df[feature_cols].sum(axis=1) > 0).mean()) if feature_cols else 0.0
    feature_count = len(feature_cols)
    dilution_index = feature_count / cov["captured_lanes"] if cov["captured_lanes"] else np.inf
    return {
        "representation": name,
        "feature_count": feature_count,
        "captured_lanes": cov["captured_lanes"],
        "missed_lanes": cov["missed_lanes"],
        "coverage_fraction": cov["coverage_fraction"],
        "mean_feature_activation": mean_feature_activation,
        "active_rows_fraction": active_rows_fraction,
        "dilution_index": dilution_index,
        "captured_residues": cov["captured_residues"],
        "missed_residues": cov["missed_residues"],
    }

summary_df = pd.DataFrame([
    representation_summary("single global-style lane", single_df),
    representation_summary("eight one-hot local tiles", onehot_df),
    representation_summary("four grouped local tiles", grouped_df),
])

summary_df.to_csv(DATA_DIR / "03_sparse_feature_representation_summary.csv", index=False)
summary_df


## 6. Visualize feature activation matrices

Rows are integers.  
Columns are features.  
A mark means the feature activates for that integer.


In [ ]:
def plot_feature_matrix(feature_df, title, filename, max_rows=120):
    feature_cols = [c for c in feature_df.columns if c.startswith("F_")]
    M = feature_df.loc[:max_rows-1, feature_cols].to_numpy().T

    plt.figure(figsize=(12, max(2.5, 0.45 * len(feature_cols))))
    plt.imshow(M, aspect="auto", interpolation="nearest")
    plt.yticks(range(len(feature_cols)), feature_cols)
    plt.xlabel("integer row index")
    plt.ylabel("feature")
    plt.title(title)
    save_current(FIGURES_DIR / filename)
    plt.show()

plot_feature_matrix(single_df, "Feature matrix: single global-style lane", "10_feature_matrix_single_global.png")


In [ ]:
plot_feature_matrix(onehot_df, "Feature matrix: eight one-hot local residue tiles", "11_feature_matrix_one_hot_tiles.png")


In [ ]:
plot_feature_matrix(grouped_df, "Feature matrix: four grouped residue-tile detectors", "12_feature_matrix_grouped_tiles.png")


## 7. Coverage vs sparsity tradeoff

This plot separates:

- feature count
- coverage
- mean per-feature activation

The key point is not that one representation is universally best.  
The point is that “sparse + interpretable” can still be incomplete.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(summary_df["feature_count"], summary_df["coverage_fraction"], s=120)

for _, row in summary_df.iterrows():
    plt.annotate(row["representation"], (row["feature_count"], row["coverage_fraction"]),
                 textcoords="offset points", xytext=(6, 6), ha="left")

plt.xlabel("feature count")
plt.ylabel("coverage fraction of persisting Mod30 lanes")
plt.ylim(0, 1.1)
plt.title("Coverage vs feature count: sparse capture can be incomplete")
save_current(FIGURES_DIR / "13_coverage_sparsity_tradeoff.png")
plt.show()


## 8. Fragmentation / dilution summary

This plot makes the representation tradeoff explicit.


In [ ]:
x = np.arange(len(summary_df))
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, summary_df["coverage_fraction"], width, label="coverage")
plt.bar(x, summary_df["mean_feature_activation"], width, label="mean feature activation")
plt.bar(x + width, summary_df["dilution_index"], width, label="dilution index")

plt.xticks(x, summary_df["representation"], rotation=20, ha="right")
plt.ylabel("metric value")
plt.title("Sparse-feature analogy: coverage, activation, dilution")
plt.legend()
save_current(FIGURES_DIR / "14_fragmentation_dilution_summary.png")
plt.show()


## 9. Interpretation

| Representation | What it shows |
|---|---|
| single global-style lane | legible but structurally incomplete |
| eight one-hot local tiles | fragmented but complete |
| four grouped local tiles | compact and complete at coarser resolution |

Paper-facing phrase:

```text
Sparse local detectors can look fragmented, but fragmentation can preserve persisting structure when the union of tiles is measured.
```


## 10. Save feature matrices and summary

In [ ]:
single_df.to_csv(DATA_DIR / "03_single_global_feature_matrix.csv", index=False)
onehot_df.to_csv(DATA_DIR / "03_onehot_tile_feature_matrix.csv", index=False)
grouped_df.to_csv(DATA_DIR / "03_grouped_tile_feature_matrix.csv", index=False)

summary_md = f"""# Notebook 03 Summary — Sparse Feature Analogy

Notebook 03 builds toy sparse-feature representations over Mod30 residue lanes.

## Result

{summary_df.to_markdown(index=False)}

## Interpretation

One detector is sparse and readable but incomplete.
One-hot local tiles are fragmented but complete.
Grouped local tiles trade compactness for coverage.

## Generated figures

- `figures/10_feature_matrix_single_global.png`
- `figures/11_feature_matrix_one_hot_tiles.png`
- `figures/12_feature_matrix_grouped_tiles.png`
- `figures/13_coverage_sparsity_tradeoff.png`
- `figures/14_fragmentation_dilution_summary.png`

## Generated data

- `data/03_sparse_feature_representation_summary.csv`
- `data/03_single_global_feature_matrix.csv`
- `data/03_onehot_tile_feature_matrix.csv`
- `data/03_grouped_tile_feature_matrix.csv`
"""

summary_path = OUTPUTS_DIR / "03_sparse_feature_analogy_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)


## 11. Optional: zip-download pattern

Uncomment in Colab to download figures, data, and output summaries.


In [ ]:
# Optional zip-download pattern:
#
# import shutil
#
# bundle_name = "notebook_03_sparse_feature_analogy_outputs"
# bundle_base = REPO_ROOT / bundle_name
# bundle_zip = REPO_ROOT / f"{bundle_name}.zip"
#
# if bundle_base.exists():
#     shutil.rmtree(bundle_base)
#
# bundle_base.mkdir(parents=True, exist_ok=True)
#
# for folder_name in ["figures", "data", "outputs"]:
#     src_folder = REPO_ROOT / folder_name
#     dst_folder = bundle_base / folder_name
#     if src_folder.exists():
#         shutil.copytree(src_folder, dst_folder)
#
# shutil.make_archive(str(bundle_base), "zip", bundle_base)
# print("Created:", bundle_zip)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(str(bundle_zip))


## 12. Recommended Notebook 04

```text
04_mod30_sae_toy_model.ipynb
```

Goal:

- create synthetic continuous embeddings for residue lanes
- train or simulate sparse-code recovery
- compare learned detectors against exact residue tiles
- make direct bridge to SAE-style manifold tiling
```
